In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/pykt-scripts/DKT_A_f0_preds.parquet
/kaggle/input/datasets/nhatanhadr/pykt-scripts/metrics.py
/kaggle/input/datasets/nhatanhadr/pykt-scripts/predict_sharded.py
/kaggle/input/datasets/nhatanhadr/pyk

In [2]:
%%bash
# Một ô Kaggle cho một lần chạy pyKT: huấn luyện -> predict chia shard -> registry.
# Chỉ cho mô hình đọc test bằng KTDataset (dkt, akt, simplekt, sakt...). QIKT, IEKT
# đọc các file *_quelevel.csv: liên kết thêm các file đó và dùng wandb_predict.py gốc.
set -e
cd /kaggle/working
[ -d pykt-toolkit ] || git clone -q https://github.com/pykt-team/pykt-toolkit.git
cd pykt-toolkit && git checkout -q eec087e   # ghim commit (các khung đã thử với eec087e)
pip install -q -e . && pip install -q einops entmax
X=/kaggle/input/datasets/nhatanhadr/xes3g5m-extended
T=/kaggle/input/datasets/nhatanhadr/pykt-scripts
REF=$T/DKT_A_f0_preds.parquet
mkdir -p data/peiyou
for f in train_valid_sequences test_question_window_sequences; do
  ln -sf $X/$f.csv data/peiyou/$f.csv             # liên kết, không chép: tiết kiệm đĩa
done
cp $T/*.py examples/ && cd examples && mkdir -p logs
M=dkt; F=0; S=42                                  # mô hình, fold, seed
L=logs/${M}_f${F}_s${S}
( while true; do echo "$(date +%T) $(free -g | awk '/Mem/ {print $3"/"$2}') GB"
    sleep 15; done ) > ${L}_ram.log &             # RAM mỗi 15 giây
trap "kill $! 2>/dev/null" EXIT
python -u wandb_${M}_train.py --dataset_name=peiyou --fold=$F --seed=$S \
  --use_wandb=0 --add_uuid=0 --save_dir=sm_$M > ${L}_train.log 2>&1
CKPT=$(dirname $(find sm_$M -name config.json | head -n 1))
python -u predict_sharded.py --save_dir=$CKPT --n_shards=8 --bz=64 > ${L}_predict.log 2>&1
python pykt_to_registry.py --save_dir=$CKPT --train_log=${L}_train.log \
  --test_csv=$X/test.csv --pykt_commit=eec087e --runs_root=/kaggle/working/runs \
  --ref_preds=$REF
tail -3 ${L}_predict.log; cat /kaggle/working/runs/registry.csv
echo "RAM cao nhất (GB dùng/tổng): $(awk '{print $2}' ${L}_ram.log | sort -n | tail -1)"

DKT_pyKT_f0_s42_20261025-0906 {'auc': np.float64(0.7834), 'acc': 0.8169, 'rmse': 0.3633, 'brier': np.float64(0.132), 'ece': 0.0079, 'auprc_wrong': np.float64(0.5053), 'f1_wrong': 0.3701}
3613 học sinh, 1099197 câu hỏi được chấm
windowauclate_mean = 0.783422
windowacclate_mean = 0.816918
run_id,model,variant,fold,seed,auc,acc,rmse,brier,ece,auprc_wrong,f1_wrong,n_epochs,sec_per_epoch,train_time_s,peak_mem_gb,n_params,preds_path,preds_raw_path
DKT_pyKT_f0_s42_20261025-0906,DKT,pyKT,0,42,0.783422,0.816918,0.36332,0.132001,0.007857,0.505273,0.370062,18,17.666666666666668,318.0,,841465,DKT_pyKT_f0_s42_20261025-0906/preds.parquet,DKT_pyKT_f0_s42_20261025-0906/preds_raw.parquet
RAM cao nhất (GB dùng/tổng): 6/31
